### **Investigación opcional sobre razonamiento multimodal y grounding**

#### **Multimodal CoT, verificación visual, HallusionBench adaptado y benchmark de grounding por dominio**

Este cuaderno propone tres líneas de investigación opcional. No reemplaza el cuaderno 16. Funciona como una apertura metodológica para estudiantes que quieren convertir el análisis de alucinación, grounding y consistencia en un experimento de investigación aplicada.

Las tres líneas son:

1. Multimodal CoT con verificación visual incorporada.
2. Diagnóstico de falla en razonamiento multimodal con un esquema inspirado en HallusionBench.
3. Benchmark de grounding para un dominio específico.

El cuaderno es ejecutable sin GPU porque usa datos simulados y verificadores deterministas. Las secciones avanzadas indican dónde conectar modelos reales como VLMs, Grounding DINO o datasets externos.

### **Objetivos de investigación**

#### **Preguntas principales**

1. ¿Separar el razonamiento en etapas mejora la respuesta o solo produce racionales más extensos?
2. ¿Una etapa de verificación visual reduce alucinaciones en el racional antes de emitir la conclusión?
3. ¿Qué tipo de razonamiento multimodal es más vulnerable a alucinación: deductivo, abductivo o analógico?
4. ¿Cómo se diseña un benchmark de grounding para un dominio específico sin confundir exactitud textual con evidencia visual?
5. ¿Qué limitaciones aparecen cuando se usan verificadores automáticos como sustitutos parciales de auditoría humana?.

### **Mapa del cuaderno**

#### **Estructura**

1. Configuración experimental reproducible.
2. Dataset simulado para investigación.
3. Multimodal CoT con verificación visual incorporada.
4. Diagnóstico de falla inspirado en HallusionBench.
5. Benchmark de grounding para un dominio elegido.
6. Comparación de modelos VLM modernos como diseño experimental.
7. Plantilla de propuesta de investigación.
8. Cierre metodológico.

In [ ]:
import json
import random
import platform
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

In [ ]:
# Se fija la semilla para que los ejemplos simulados sean reproducibles.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


# Se crea una carpeta de resultados local.
def make_output_dir(path_text):
    path = Path(path_text)
    path.mkdir(parents=True, exist_ok=True)
    return path


# Se documenta el entorno básico del experimento.
def get_environment_metadata():
    return {
        "python": platform.python_version(),
        "sistema": platform.platform(),
        "procesador": platform.processor(),
    }


set_seed(42)
OUTPUT_DIR = make_output_dir("results/cuaderno16b_mcc225")
get_environment_metadata()

### **Configuración experimental**

#### **Metadatos mínimos**

Todo experimento de investigación debe registrar:

1. Identificador del experimento.
2. Modelo o simulador usado.
3. Dataset o conjunto de casos.
4. Semilla aleatoria.
5. Parámetros de generación si hay un VLM real.
6. Criterios de evaluación.
7. Limitaciones del protocolo.

En este cuaderno se usa un protocolo simulado para que la clase pueda ejecutar y modificar las ideas sin depender de descargas pesadas.

In [ ]:
@dataclass
class ResearchConfig:
    experiment_id: str
    researcher: str
    seed: int
    mode: str
    domain: str
    notes: str


config = ResearchConfig(
    experiment_id="mcc225_semana10_investigacion_opcional",
    researcher="Estudiante MCC225",
    seed=42,
    mode="simulacion_reproducible",
    domain="educativo",
    notes="Protocolo opcional para abrir líneas de investigación en grounding y alucinación",
)

with open(OUTPUT_DIR / "configuracion_experimental.json", "w", encoding="utf-8") as file:
    json.dump(asdict(config), file, indent=2, ensure_ascii=False)

asdict(config)

### **Dataset simulado de investigación**

#### **Casos de prueba**

Cada caso representa una pregunta multimodal con metadatos suficientes para estudiar grounding, alucinación y tipo de razonamiento.

Campos usados:

1. `case_id`: identificador del caso.
2. `domain`: dominio del ejemplo.
3. `question`: pregunta.
4. `expected_answer`: respuesta esperada.
5. `visible_evidence`: evidencia que debería sustentar la respuesta.
6. `absent_elements`: objetos o evidencias que no están presentes.
7. `reasoning_type`: tipo de razonamiento.
8. `hallusion_category`: categoría inspirada en HallusionBench.
9. `difficulty`: dificultad del caso.

In [ ]:
def build_research_cases():
    return [
        {
            "case_id": "caso_001",
            "domain": "educativo",
            "question": "¿El diagrama muestra que la corriente aumenta cuando baja la resistencia?",
            "expected_answer": "si",
            "visible_evidence": ["ley de ohm", "resistencia menor", "corriente mayor"],
            "absent_elements": ["voltaje variable", "bateria adicional"],
            "reasoning_type": "deductivo",
            "hallusion_category": "alucinacion_linguistica",
            "difficulty": "media",
        },
        {
            "case_id": "caso_002",
            "domain": "documentos",
            "question": "¿La tabla contiene una columna llamada costo total?",
            "expected_answer": "no",
            "visible_evidence": ["precio unitario", "cantidad", "subtotal"],
            "absent_elements": ["costo total"],
            "reasoning_type": "perceptual",
            "hallusion_category": "ilusion_visual",
            "difficulty": "baja",
        },
        {
            "case_id": "caso_003",
            "domain": "industrial",
            "question": "¿La pieza defectuosa está a la izquierda del sensor?",
            "expected_answer": "si",
            "visible_evidence": ["pieza defectuosa", "sensor", "relacion izquierda"],
            "absent_elements": ["alarma encendida"],
            "reasoning_type": "espacial",
            "hallusion_category": "ambos",
            "difficulty": "alta",
        },
        {
            "case_id": "caso_004",
            "domain": "medico_simulado",
            "question": "¿La imagen muestra evidencia suficiente para afirmar fractura?",
            "expected_answer": "no",
            "visible_evidence": ["imagen ambigua", "no hay linea clara"],
            "absent_elements": ["fractura confirmada"],
            "reasoning_type": "abductivo",
            "hallusion_category": "alucinacion_linguistica",
            "difficulty": "alta",
        },
        {
            "case_id": "caso_005",
            "domain": "legal_simulado",
            "question": "¿El documento contiene una firma manuscrita visible?",
            "expected_answer": "si",
            "visible_evidence": ["firma manuscrita", "bloque de aprobacion"],
            "absent_elements": ["sello notarial"],
            "reasoning_type": "perceptual",
            "hallusion_category": "ilusion_visual",
            "difficulty": "media",
        },
        {
            "case_id": "caso_006",
            "domain": "educativo",
            "question": "¿La gráfica apoya la conclusión de crecimiento lineal?",
            "expected_answer": "no",
            "visible_evidence": ["curva no lineal", "pendiente cambiante"],
            "absent_elements": ["recta ajustada"],
            "reasoning_type": "analógico",
            "hallusion_category": "ambos",
            "difficulty": "alta",
        },
    ]


cases = build_research_cases()
cases_df = pd.DataFrame(cases)
cases_df

### **Línea 1: Multimodal CoT con verificación visual incorporada**

#### **Idea central**

La generación directa puede responder sin justificar. Multimodal CoT intenta separar observación, interpretación, razonamiento y conclusión. El problema es que el racional también puede alucinar. Por eso se agrega una etapa intermedia de verificación visual:

1. El VLM genera una respuesta candidata.
2. El protocolo extrae claims visuales verificables.
3. Un verificador visual revisa si cada claim tiene evidencia.
4. El revisor decide si la conclusión debe mantenerse, corregirse o declararse insuficiente.

In [ ]:
@dataclass
class VisualClaim:
    text: str
    claim_type: str
    source: str


def simulate_mcot_response(case):
    # Esta función simula una respuesta con racional estructurado.
    evidence = ", ".join(case["visible_evidence"])
    absent = case["absent_elements"][0]

    if case["difficulty"] == "alta":
        hallucinated_claim = f"tambien se observa {absent}"
    else:
        hallucinated_claim = "no se agrega evidencia inexistente"

    return {
        "observacion": f"Se observan elementos relevantes: {evidence}.",
        "razonamiento": f"La respuesta se infiere a partir de la evidencia visible. {hallucinated_claim}.",
        "conclusion": case["expected_answer"],
    }


def extract_visual_claims(response_parts):
    # Se extraen claims simples mediante reglas deterministas.
    text = " ".join(response_parts.values()).lower()
    claims = []

    markers = ["se observan", "se observa", "tambien se observa", "evidencia visible"]
    for marker in markers:
        if marker in text:
            claims.append(
                VisualClaim(
                    text=marker,
                    claim_type="evidencia_visual",
                    source="racional",
                )
            )

    return claims


def verify_claims(case, response_parts):
    # Se verifica si el racional menciona evidencia que no está disponible.
    joined_response = " ".join(response_parts.values()).lower()
    verifications = []

    for element in case["visible_evidence"]:
        verifications.append({
            "claim": element,
            "verified": element.lower() in joined_response or element in case["visible_evidence"],
            "reason": "evidencia esperada del caso",
        })

    for element in case["absent_elements"]:
        appears = element.lower() in joined_response
        verifications.append({
            "claim": element,
            "verified": not appears,
            "reason": "elemento ausente mencionado" if appears else "elemento ausente no mencionado",
        })

    return verifications


def summarize_verification(verifications):
    total = len(verifications)
    supported = sum(1 for item in verifications if item["verified"])
    unsupported = total - supported
    return {
        "total_claims": total,
        "supported_claims": supported,
        "unsupported_claims": unsupported,
        "verification_ratio": supported / total if total else 0.0,
    }


mcot_rows = []
for case in cases:
    response = simulate_mcot_response(case)
    verifications = verify_claims(case, response)
    summary = summarize_verification(verifications)

    mcot_rows.append({
        "case_id": case["case_id"],
        "reasoning_type": case["reasoning_type"],
        "difficulty": case["difficulty"],
        "answer": response["conclusion"],
        "verification_ratio": summary["verification_ratio"],
        "unsupported_claims": summary["unsupported_claims"],
        "rationale": " ".join(response.values()),
    })

mcot_df = pd.DataFrame(mcot_rows)
mcot_df

### **Discusión de la línea 1**

#### **Preguntas de investigación**

1. ¿La etapa de racional mejora la respuesta o solo aumenta la longitud de salida?
2. ¿Cuántos claims del racional son verificables?
3. ¿Cuántos claims verificables carecen de evidencia visual?
4. ¿La conclusión cambia cuando se eliminan claims no verificados?
5. ¿Qué tipos de casos inducen más alucinación en el racional?.

#### **Extensión avanzada**

En una versión con modelos reales, se puede reemplazar el simulador por:

1. LLaVA-CoT o un VLM equivalente para generar etapas.
2. Un extractor de claims basado en reglas, NER o un juez LLM.
3. Grounding DINO para verificar objetos, atributos, ubicación o relaciones.
4. Una auditoría humana para revisar casos críticos.

### **Línea 2: Diagnóstico de falla inspirado en HallusionBench**

#### **Idea central**

HallusionBench permite distinguir entre fallas inducidas por sesgo lingüístico y fallas inducidas por ambigüedad visual. En un dominio propio, la adaptación puede organizarse en pares de casos:

1. Caso fácil y caso difícil.
2. Imagen clara e imagen ambigua.
3. Pregunta neutral y pregunta con sesgo lingüístico.
4. Distractor visual ausente o presente.
5. Respuesta con evidencia y respuesta por expectativa previa.

In [ ]:
def build_hallusion_style_pairs(cases):
    # Se construye una vista simplificada de pares de diagnóstico.
    pairs = []

    for case in cases:
        easy_case = {
            "pair_id": f"{case['case_id']}_facil",
            "base_case": case["case_id"],
            "category": case["hallusion_category"],
            "reasoning_type": case["reasoning_type"],
            "difficulty": "facil",
            "expected_answer": case["expected_answer"],
            "distractor": "sin_distractor",
        }

        hard_case = {
            "pair_id": f"{case['case_id']}_dificil",
            "base_case": case["case_id"],
            "category": case["hallusion_category"],
            "reasoning_type": case["reasoning_type"],
            "difficulty": "dificil",
            "expected_answer": case["expected_answer"],
            "distractor": "con_distractor",
        }

        pairs.extend([easy_case, hard_case])

    return pairs


def simulate_model_on_hallusion_pair(pair):
    # Se simula un modelo más vulnerable en casos difíciles y abductivos.
    risk = 0.15

    if pair["difficulty"] == "dificil":
        risk += 0.25
    if pair["category"] == "alucinacion_linguistica":
        risk += 0.20
    if pair["reasoning_type"] == "abductivo":
        risk += 0.20
    if pair["reasoning_type"] == "analógico":
        risk += 0.10

    is_error = np.random.random() < risk

    return {
        **pair,
        "model_answer": "no" if pair["expected_answer"] == "si" and is_error else pair["expected_answer"],
        "is_correct": not is_error,
        "estimated_risk": round(risk, 3),
    }


hallusion_pairs = build_hallusion_style_pairs(cases)
hallusion_results = [simulate_model_on_hallusion_pair(pair) for pair in hallusion_pairs]
hallusion_df = pd.DataFrame(hallusion_results)
hallusion_df

In [ ]:
def analyze_failure_by_category(results_df):
    grouped = (
        results_df
        .groupby(["category", "reasoning_type"])
        .agg(
            total=("is_correct", "size"),
            correct=("is_correct", "sum"),
            mean_risk=("estimated_risk", "mean"),
        )
        .reset_index()
    )
    grouped["accuracy"] = grouped["correct"] / grouped["total"]
    grouped["failure_rate"] = 1.0 - grouped["accuracy"]
    return grouped.sort_values(["failure_rate", "mean_risk"], ascending=False)


failure_by_category_df = analyze_failure_by_category(hallusion_df)
failure_by_category_df

### **Discusión de la línea 2**

#### **Preguntas de investigación**

1. ¿Qué falla más: razonamiento deductivo, abductivo, analógico, espacial o perceptual?
2. ¿La dificultad visual aumenta la alucinación o solo baja la exactitud?
3. ¿La pregunta sesgada cambia la respuesta cuando la imagen no la respalda?
4. ¿Qué casos requieren auditoría humana porque el error automático no basta?
5. ¿Qué categorías deberían ampliarse en un benchmark real?.

#### **Uso en tesis o artículo**

Esta línea puede convertirse en un diagnóstico por dominio si se definen pares cuidadosamente anotados. Lo importante no es tener muchos ejemplos al inicio, sino que cada par tenga una hipótesis clara sobre el modo de falla.

### **Línea 3: Benchmark de grounding para un dominio específico**

#### **Idea central**

Un benchmark de grounding no pregunta solo si la respuesta textual es correcta. Pregunta si el modelo puede localizar, citar o señalar la evidencia que justifica la respuesta.

Para un dominio como medicina, derecho, industria o educación, cada caso debería incluir:

1. Entrada multimodal.
2. Pregunta.
3. Respuesta esperada.
4. Evidencia esperada.
5. Región, fragmento o elemento visual relevante.
6. Tipo de evidencia.
7. Riesgo si la evidencia es inventada.

In [ ]:
def build_domain_grounding_suite(domain):
    # Se crea una plantilla pequeña para diseñar un benchmark por dominio.
    templates = {
        "educativo": [
            {
                "case_id": "edu_001",
                "question": "¿La solución aplica correctamente la ley indicada?",
                "expected_answer": "no",
                "expected_evidence": ["formula escrita", "sustitucion incorrecta"],
                "evidence_type": "texto_en_imagen",
                "risk_level": "medio",
            },
            {
                "case_id": "edu_002",
                "question": "¿El gráfico apoya la conclusión del estudiante?",
                "expected_answer": "no",
                "expected_evidence": ["curva no lineal", "conclusion lineal"],
                "evidence_type": "grafico",
                "risk_level": "alto",
            },
        ],
        "industrial": [
            {
                "case_id": "ind_001",
                "question": "¿La zona marcada coincide con el defecto visible?",
                "expected_answer": "si",
                "expected_evidence": ["grieta", "zona marcada"],
                "evidence_type": "region_visual",
                "risk_level": "alto",
            },
        ],
        "documentos": [
            {
                "case_id": "doc_001",
                "question": "¿El documento muestra una fecha de vencimiento?",
                "expected_answer": "si",
                "expected_evidence": ["fecha de vencimiento", "campo de documento"],
                "evidence_type": "ocr",
                "risk_level": "medio",
            },
        ],
    }

    return templates.get(domain, templates["educativo"])


def simulate_grounding_prediction(case):
    # Se simula una predicción de grounding con evidencia parcial.
    evidence = case["expected_evidence"]
    if case["risk_level"] == "alto":
        predicted_evidence = evidence[:1]
    else:
        predicted_evidence = evidence

    return {
        "case_id": case["case_id"],
        "expected_answer": case["expected_answer"],
        "predicted_answer": case["expected_answer"],
        "expected_evidence": evidence,
        "predicted_evidence": predicted_evidence,
        "risk_level": case["risk_level"],
    }


def compute_grounding_score(row):
    expected = set(row["expected_evidence"])
    predicted = set(row["predicted_evidence"])

    if not expected:
        return 0.0

    return len(expected & predicted) / len(expected)


domain_suite = build_domain_grounding_suite("educativo")
grounding_predictions = [simulate_grounding_prediction(case) for case in domain_suite]
grounding_df = pd.DataFrame(grounding_predictions)
grounding_df["grounding_score"] = grounding_df.apply(compute_grounding_score, axis=1)
grounding_df

### **Diseño comparativo de modelos VLM modernos**

#### **Uso como plan experimental**

Una comparación real puede incluir modelos como Qwen2.5-VL, InternVL 2.5, PaliGemma 2 o LLaVA-OneVision. Para que la comparación sea metodológicamente defendible, se debe controlar:

1. El mismo conjunto de casos.
2. El mismo prompt base.
3. La misma temperatura o modo determinista.
4. La misma definición de respuesta correcta.
5. La misma definición de evidencia válida.
6. El mismo protocolo de auditoría.
7. El costo computacional y la latencia.

En este cuaderno no se cargan modelos reales. Se deja una tabla de diseño para preparar el experimento.

In [ ]:
def build_model_comparison_plan():
    return pd.DataFrame([
        {
            "model_id": "qwen2_5_vl",
            "family": "Qwen2.5-VL",
            "candidate_strength": "localizacion y agente visual",
            "risk_to_audit": "puede producir racionales sobreconfiados",
            "status": "opcional",
        },
        {
            "model_id": "internvl_2_5",
            "family": "InternVL 2.5",
            "candidate_strength": "multiimagen y escalamiento en inferencia",
            "risk_to_audit": "sensibilidad a configuracion de evaluacion",
            "status": "opcional",
        },
        {
            "model_id": "paligemma_2",
            "family": "PaliGemma 2",
            "candidate_strength": "OCR y transferencia eficiente",
            "risk_to_audit": "fallas en documentos visualmente complejos",
            "status": "opcional",
        },
        {
            "model_id": "llava_onevision",
            "family": "LLaVA-OneVision",
            "candidate_strength": "unificacion imagen y video",
            "risk_to_audit": "consistencia entre modalidades",
            "status": "opcional",
        },
    ])


model_plan_df = build_model_comparison_plan()
model_plan_df

### **Plantilla de propuesta de investigación**

#### **Uso**

La siguiente celda genera una plantilla breve para que el estudiante convierta una de las tres líneas en una propuesta de investigación.

In [ ]:
def build_research_proposal_template():
    return {
        "titulo": "Completar titulo del estudio",
        "linea": "mcot_verificacion_visual | diagnostico_hallusionbench | benchmark_grounding_dominio",
        "problema": "Describir el problema de alucinacion, grounding o razonamiento que se quiere estudiar.",
        "pregunta_investigacion": "Formular una pregunta especifica y verificable.",
        "hipotesis": [
            "Hipotesis 1",
            "Hipotesis 2",
        ],
        "modelo_o_sistema": "Indicar VLM, simulador o arquitectura a evaluar.",
        "datos": {
            "fuente": "Dataset publico, dataset propio o casos sinteticos",
            "tamano_inicial": "Numero de casos piloto",
            "criterios_seleccion": "Criterios para incluir o excluir casos",
        },
        "metodo": [
            "Definir prompts",
            "Generar respuestas",
            "Extraer claims",
            "Verificar evidencia",
            "Analizar errores",
            "Revisar casos criticos manualmente",
        ],
        "metricas": [
            "exactitud",
            "tasa de alucinacion",
            "grounding_score",
            "consistencia por prompt",
            "claims no verificados",
        ],
        "limitaciones": [
            "El verificador automatico tambien puede fallar",
            "El dataset piloto puede no generalizar",
            "Las respuestas ambiguas requieren revision humana",
        ],
        "resultado_esperado": "Indicar que evidencia permitiria apoyar o rechazar la hipotesis.",
    }


proposal_template = build_research_proposal_template()
with open(OUTPUT_DIR / "plantilla_propuesta_investigacion.json", "w", encoding="utf-8") as file:
    json.dump(proposal_template, file, indent=2, ensure_ascii=False)

proposal_template

### **Persistencia de resultados**

#### **Archivos generados**

La siguiente celda guarda tablas útiles para continuar el trabajo fuera del cuaderno.

In [ ]:
mcot_df.to_csv(OUTPUT_DIR / "mcot_verificacion_visual.csv", index=False)
hallusion_df.to_csv(OUTPUT_DIR / "hallusionbench_adaptado_simulado.csv", index=False)
failure_by_category_df.to_csv(OUTPUT_DIR / "diagnostico_falla_por_categoria.csv", index=False)
grounding_df.to_csv(OUTPUT_DIR / "benchmark_grounding_dominio.csv", index=False)
model_plan_df.to_csv(OUTPUT_DIR / "plan_comparativo_modelos_vlm.csv", index=False)

summary = {
    "num_cases": len(cases_df),
    "avg_mcot_verification_ratio": float(mcot_df["verification_ratio"].mean()),
    "avg_hallusion_failure_rate": float(1.0 - hallusion_df["is_correct"].mean()),
    "avg_domain_grounding_score": float(grounding_df["grounding_score"].mean()),
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "resumen_experimento_opcional.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

summary

#### **Conclusión**

Este cuaderno abre tres rutas de investigación de nivel posgrado:

1. Agregar verificación visual a Multimodal CoT para auditar racionales antes de aceptar conclusiones.
2. Adaptar un diagnóstico tipo HallusionBench para separar alucinación lingüística, ilusión visual y fallas combinadas.
3. Construir un benchmark de grounding para un dominio específico donde la evidencia sea tan importante como la respuesta.

La contribución esperada no es solo obtener una métrica alta. La contribución consiste en diseñar un protocolo que permita decir con precisión qué evidencia tiene el modelo, qué inventa, qué no puede verificar y bajo qué condiciones sus conclusiones dejan de ser confiables.